# Phase 3: Model Training

## Objective

In this phase, we train machine learning models to forecast daily sales quantity for each SKU.

The workflow includes:

- Train/Test Split
- Baseline Model (Random Forest)
- LightGBM Model
- Hyperparameter Tuning with Optuna
- Model Evaluation
- Save Trained Model

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgbm

# Evaluation Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


In [2]:
import sys
sys.path.append("../")

from src.utils.utils import weighted_absolute_percentage_error

In [3]:
data = pd.read_csv(
    "../data/processed/data_engineered.csv",
    parse_dates=["shipped_date"]
)

data.head()

,shipped_date,sku,channel,qty,revenue,COGS,MOQ_orders,month,day,dayofweek,...,mean_qty_sku_dow,mean_qty_sku_month,qty_log,lag1,lag7,lag30,rolling_mean_7,rolling_mean_30,rolling_std_7,rolling_std_30
0,2021-01-03,017SAD,ADS,75,3358.95,1660.05,1425,1,3,6,...,0.0,0.0,4.330733,NaN,NaN,NaN,75.0,75.0,NaN,NaN
1,2021-01-15,017SAD,ADS,75,3358.95,1660.05,1425,1,15,4,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
2,2021-02-16,017SAD,ADS,75,3358.95,1660.05,1425,2,16,1,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
3,2021-02-16,017SAD,LAL,75,2310.00,2009.70,1425,2,16,1,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
4,2021-02-26,017SAD,ADS,75,3358.95,1660.05,1425,2,26,4,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0


In [4]:
data.info()
print(data.shape)

<class 'pandas.DataFrame'>
RangeIndex: 48345 entries, 0 to 48344
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   shipped_date        48345 non-null  datetime64[us]
 1   sku                 48345 non-null  str           
 2   channel             48345 non-null  str           
 3   qty                 48345 non-null  int64         
 4   revenue             48345 non-null  float64       
 5   COGS                48345 non-null  float64       
 6   MOQ_orders          48345 non-null  int64         
 7   month               48345 non-null  int64         
 8   day                 48345 non-null  int64         
 9   dayofweek           48345 non-null  int64         
 10  weekofyear          48345 non-null  int64         
 11  quarter             48345 non-null  int64         
 12  is_weekend          48345 non-null  int64         
 13  is_even_day         48345 non-null  int64         
 14  i

## Train-Test Split

Since this is a forecasting problem, the dataset is split chronologically instead of randomly.

In [5]:
data_model = data.sort_values("shipped_date")

split_idx = int(len(data_model) * 0.8)

train = data_model.iloc[:split_idx]

test = data_model.iloc[split_idx:]

print(train.shape)

print(test.shape)

(38676, 26)
(9669, 26)


In [6]:
X_train = train.drop(columns=["qty_log"])
y_train = train["qty_log"]

X_test = test.drop(columns=["qty_log"])
y_test = test["qty_log"]

In [11]:
drop_cols = [
    "qty",
    "shipped_date",
    "sku",
    "revenue",
    "COGS",
    "channel",
    "MOQ_orders"
]

X_train = X_train.drop(columns=drop_cols)

X_test = X_test.drop(columns=drop_cols)

KeyError: "['qty', 'shipped_date', 'sku', 'revenue', 'COGS'] not found in axis"

In [12]:
X_train = X_train.drop(columns=["channel", "MOQ_orders"])
X_test = X_test.drop(columns=["channel", "MOQ_orders"])

In [8]:
print(X_train.shape)
print(X_test.shape)

X_train.head()

(38676, 20)
(9669, 20)


,channel,MOQ_orders,month,day,dayofweek,weekofyear,quarter,is_weekend,is_even_day,is_holiday,dow,mean_qty_sku_dow,mean_qty_sku_month,lag1,lag7,lag30,rolling_mean_7,rolling_mean_30,rolling_std_7,rolling_std_30
27860,ADS,31521,1,1,4,53,1,0,0,1,4,0.0,0.0,NaN,NaN,NaN,57.0,57.0,NaN,NaN
19816,LAL,54080,1,1,4,53,1,0,0,1,4,455.0,455.0,455.0,NaN,NaN,260.0,260.0,275.771645,275.771645
47743,ADS,83844,1,1,4,53,1,0,0,1,4,0.0,0.0,NaN,NaN,NaN,306.0,306.0,NaN,NaN
27540,ADS,27897,1,1,4,53,1,0,0,1,4,68.0,68.0,68.0,NaN,NaN,59.5,59.5,12.020815,12.020815
27539,AWH,27897,1,1,4,53,1,0,0,1,4,0.0,0.0,NaN,NaN,NaN,68.0,68.0,NaN,NaN


## Baseline Model: Random Forest

Before training the final LightGBM model, a Random Forest Regressor is used as a baseline model to evaluate forecasting performance.

This provides a benchmark for comparing more advanced models.

### Train Random Forest

In [13]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

### Prediction

In [14]:
rf_pred_log = rf_model.predict(X_test)

rf_pred = np.expm1(rf_pred_log)

actual = np.expm1(y_test)

### Evaluation

In [15]:
rf_mae = mean_absolute_error(actual, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(actual, rf_pred)
)

rf_r2 = r2_score(actual, rf_pred)

rf_wape = weighted_absolute_percentage_error(
    actual,
    rf_pred
)

In [16]:
print("=" * 50)
print("Random Forest Performance")
print("=" * 50)

print(f"MAE  : {rf_mae:.2f}")
print(f"RMSE : {rf_rmse:.2f}")
print(f"R²   : {rf_r2:.4f}")
print(f"WAPE : {rf_wape:.2f}%")

Random Forest Performance
MAE  : 115.64
RMSE : 224.44
R²   : 0.5458
WAPE : 49.83%


### Baseline Model Performance

The Random Forest model serves as a baseline for comparison.

Although it captures part of the sales pattern (R² ≈ 0.55), the prediction error remains relatively high (WAPE ≈ 50%), indicating room for improvement.

In the next section, LightGBM will be trained to improve forecasting performance.

### Time Series Cross Validation

In [18]:
import optuna

In [20]:
from sklearn.model_selection import TimeSeriesSplit

In [21]:
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, valid_idx) in enumerate(tscv.split(X_train)):
    print(
        f"Fold {fold+1}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(valid_idx)}"
    )

Fold 1: Train=6446, Validation=6446
Fold 2: Train=12892, Validation=6446
Fold 3: Train=19338, Validation=6446
Fold 4: Train=25784, Validation=6446
Fold 5: Train=32230, Validation=6446


In [22]:
def objective(trial):

    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 42,

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            100
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            5,
            50
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "n_estimators": 500
    }

    rmse_scores = []

    for train_idx, valid_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[valid_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        model = lgbm.LGBMRegressor(**params)

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                pred
            )
        )

        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

### Study

In [23]:
study = optuna.create_study(
    direction="minimize",
    study_name="lightgbm_sales_forecasting"
)

[I 2026-07-08 14:18:21,984] A new study created in memory with name: lightgbm_sales_forecasting


In [24]:
study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-08 14:18:39,113] Trial 0 finished with value: 0.6487984284452335 and parameters: {'learning_rate': 0.09447164286340702, 'num_leaves': 85, 'max_depth': 3, 'min_child_samples': 45, 'subsample': 0.9297419280122075, 'colsample_bytree': 0.8264268401300149}. Best is trial 0 with value: 0.6487984284452335.
[I 2026-07-08 14:18:47,789] Trial 1 finished with value: 0.6493167667057654 and parameters: {'learning_rate': 0.06357678207832207, 'num_leaves': 86, 'max_depth': 6, 'min_child_samples': 37, 'subsample': 0.7379399819490513, 'colsample_bytree': 0.6261016932482658}. Best is trial 0 with value: 0.6487984284452335.
[I 2026-07-08 14:18:56,915] Trial 2 finished with value: 0.6582076063985174 and parameters: {'learning_rate': 0.10451947837400373, 'num_leaves': 36, 'max_depth': 8, 'min_child_samples': 34, 'subsample': 0.6880238966853982, 'colsample_bytree': 0.91950752980603}. Best is trial 0 with value: 0.6487984284452335.
[I 2026-07-08 14:19:11,588] Trial 3 finished with value: 0.6535538

In [25]:
print("Best RMSE:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best RMSE:
0.6437659104349172

Best Parameters:
{'learning_rate': 0.011952027720200693, 'num_leaves': 53, 'max_depth': 10, 'min_child_samples': 23, 'subsample': 0.7568488016940612, 'colsample_bytree': 0.7801361637964015}


In [26]:
best_params = study.best_params

best_params

{'learning_rate': 0.011952027720200693,
 'num_leaves': 53,
 'max_depth': 10,
 'min_child_samples': 23,
 'subsample': 0.7568488016940612,
 'colsample_bytree': 0.7801361637964015}

In [33]:
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "random_state": 42,
    "verbosity": -1,
    "n_estimators": 500
})

In [34]:
final_model = lgbm.LGBMRegressor(
    **best_params
)

final_model.fit(
    X_train,
    y_train
)

,num_leaves,53
,max_depth,10
,learning_rate,0.011952027720200693
,n_estimators,500
,objective,'regression'
,min_child_samples,23
,subsample,0.7568488016940612
,colsample_bytree,0.7801361637964015
,random_state,42
,metric,'rmse'
,verbosity,-1


In [35]:
pred_log = final_model.predict(X_test)

prediction = np.expm1(pred_log)

actual = np.expm1(y_test)

In [36]:
lgb_mae = mean_absolute_error(actual, prediction)

lgb_rmse = np.sqrt(
    mean_squared_error(actual, prediction)
)

lgb_r2 = r2_score(actual, prediction)

lgb_wape = weighted_absolute_percentage_error(
    actual,
    prediction
)

In [37]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "LightGBM"],
    "MAE": [rf_mae, lgb_mae],
    "RMSE": [rf_rmse, lgb_rmse],
    "R2": [rf_r2, lgb_r2],
    "WAPE": [rf_wape, lgb_wape]
})

comparison

,Model,MAE,RMSE,R2,WAPE
0,Random Forest,115.644108,224.444922,0.545837,49.825053
1,LightGBM,111.801616,216.350784,0.578003,48.169522


### Model Comparison

LightGBM achieved better forecasting performance than the Random Forest baseline across all evaluation metrics.

Compared with Random Forest:

- MAE decreased from **115.64** to **111.80**
- RMSE decreased from **224.44** to **216.35**
- R² improved from **0.546** to **0.578**
- WAPE decreased from **49.83%** to **48.17%**

Therefore, the optimized LightGBM model was selected as the final forecasting model.

In [38]:
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/sales_forecast_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Model saved successfully!")

Model saved successfully!


In [39]:
import os

os.listdir("../models")

['sales_forecast_model.pkl']